# 반도체 공정 스케줄링 — 유전 알고리즘 + SimPy 시뮬레이션

## 개요

| 항목 | 내용 |
|---|---|
| 알고리즘 | 유전 알고리즘 (Genetic Algorithm) |
| 시뮬레이터 | SimPy (이산 사건 시뮬레이션) |
| 목표 | Q-time 위반 최소화, Makespan 최소화, Job 완료율 최대화 |
| 염색체 구조 | 순서 염색체 (job-based) + 장비 선택 염색체 |

### 염색체 구조
```
순서 염색체      : ['J1','J2','J1','J3','J2','J3']  ← job_id를 op 수만큼 반복
장비 선택 염색체 : {('J1',1):'M1', ('J1',2):'M3', ('J2',1):'M2', ...}
```

### 적합도 함수
```
Fitness = 0.50 × completed_rate        (완료된 job 비율, 최우선)
        + 0.30 × makespan_score        (1 / (1 + makespan/simul_time))
        + 0.20 × (1 - violation_rate)  (q-time 위반 페널티)

범위: 0.0 (최악) ~ 1.0 (최선)
```

---
## 1. 환경 설정

In [1]:
import os
import sys

# 프로젝트 루트 경로 설정
_notebook_dir = os.path.abspath('')
_project_root = os.path.dirname(os.path.dirname(_notebook_dir))
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# ===================== 파라미터 설정 =====================
BASE_DATA_PATH  = os.path.join(_project_root, 'schema')
SIMUL_TIME      = 100    # 시뮬레이션 시간 상한
POP_SIZE        = 10     # 개체군 크기
N_GEN           = 5      # 세대 수
CROSSOVER_RATE  = 0.85   # 교차율
MUTATION_RATE   = 0.15   # 돌연변이율
TOURNAMENT_SIZE = 3      # 토너먼트 크기
N_SIM_RUNS      = 1      # 염색체당 시뮬레이션 반복 횟수
# ========================================================

print(f'프로젝트 루트  : {_project_root}')
print(f'데이터 경로    : {BASE_DATA_PATH}')
print(f'시뮬레이션 시간: {SIMUL_TIME}')
print(f'개체군 크기    : {POP_SIZE}')
print(f'세대 수        : {N_GEN}')

프로젝트 루트  : c:\Users\User\OneDrive\대학\4-1\전종설\연구\git\simulation_teamwork
데이터 경로    : c:\Users\User\OneDrive\대학\4-1\전종설\연구\git\simulation_teamwork\schema
시뮬레이션 시간: 100
개체군 크기    : 10
세대 수        : 5


---
## 2. 모듈 import

In [2]:
from simulation import DataLoader
from algorithms.genetic.ga_simul_04 import GeneticAlgorithmWithSim

---
## 3. 데이터 로드

In [3]:
data_loader = DataLoader(BASE_DATA_PATH)
data = data_loader.load_all_data()

print('=' * 60)
print('데이터 개요')
print('=' * 60)
print(f"Jobs                 : {len(data['jobs'])} 개")
print(f"Operations           : {len(data['operations'])} 개")
print(f"Machines             : {len(data['machines'])} 개")
print(f"Machine Failures     : {len(data['machine_failure'])} 개")
print(f"Setup Times          : {len(data['setup_times'])} 개")
print(f"Operation-Machine Map: {len(data['operation_machine_map'])} 개")

print()
print('--- Jobs ---')
print(data['jobs'].to_string(index=False))

print()
print('--- Machines ---')
print(data['machines'].to_string(index=False))

데이터 개요
Jobs                 : 10 개
Operations           : 35 개
Machines             : 8 개
Machine Failures     : 8 개
Setup Times          : 12 개
Operation-Machine Map: 95 개

--- Jobs ---
job_id job_type  release_time  due_date  priority
    J1       P1             0       120         1
    J2       P2             0       130         1
    J3       P1             5       140         2
    J4       P2             5       135         1
    J5       P1            10       150         2
    J6       P2            15       160         1
    J7       P1            20       155         2
    J8       P2            20       170         1
    J9       P1            25       165         1
   J10       P2            30       180         2

--- Machines ---
machine_id machine_group
        M1            G1
        M2            G1
        M3            G1
        M4            G2
        M5            G2
        M6            G3
        M7            G3
        M8            G3


---
## 4. 유전 알고리즘 실행

각 세대마다 현재 최적 염색체를 SimPy로 시뮬레이션하여
공정 로그와 KPI를 출력합니다.

In [4]:
ga = GeneticAlgorithmWithSim(
    data=data,
    population_size=POP_SIZE,
    n_generations=N_GEN,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    simul_time=SIMUL_TIME,
    n_sim_runs=N_SIM_RUNS,
)

In [5]:
best = ga.run()

 반도체 공정 스케줄링 유전 알고리즘
  개체군 크기      : 10
  세대 수          : 5
  교차율           : 0.85
  돌연변이율       : 0.15
  시뮬레이션 시간  : 100
  시뮬 반복 횟수   : 1
  Job 수           : 10
  총 Operation 수  : 35

[세대 1 평가 중...] 

완료

  세대 1 / 5

[공정 시뮬레이션 - SimPy]
------------------------------------------------------------
0.0	Job J1 starts setup for operation J1_O1 on machine M3
0.0	Job J1 starts processing operation J1_O1 on machine M3
0.01	Job J2 starts setup for operation J2_O1 on machine M1
0.01	Job J2 starts processing operation J2_O1 on machine M1
5.01	Job J3 starts setup for operation J3_O1 on machine M2
5.01	Job J3 starts processing operation J3_O1 on machine M2
8.01	Job J2 finished operation J2_O1 on machine M1
8.01	Job J2 starts setup for operation J2_O2 on machine M5
8.01	Job J4 starts setup for operation J4_O1 on machine M1
8.01	Job J2 starts processing operation J2_O2 on machine M5
8.01	Job J4 starts processing operation J4_O1 on machine M1
16.0	Job J1 finished operation J1_O1 on machine M3
16.0	Job J1 starts setup for operation J1_O2 on machine M4
16.0	Job J5 starts setup for operation J5_O1 on machine M3
16.0	Job J1 starts processing operation J1_O2 on machine M4
16.0	Job J5 starts processing o

---
## 5. 최종 최적 염색체 상세 정보

In [6]:
print('=' * 60)
print('최적 순서 염색체 (sequence)')
print('=' * 60)
print(best.sequence)

print()
print('=' * 60)
print('최적 장비 선택 염색체 (machine_assignment)')
print('=' * 60)
for (job_id, op_seq), machine_id in sorted(best.machine_assignment.items()):
    print(f'  Job {job_id}  Op {op_seq}  →  {machine_id}')

print()
print(f'최종 적합도 : {round(best.fitness, 4)}')

최적 순서 염색체 (sequence)
['J9', 'J1', 'J3', 'J1', 'J10', 'J6', 'J5', 'J10', 'J8', 'J4', 'J10', 'J5', 'J8', 'J4', 'J3', 'J3', 'J2', 'J3', 'J2', 'J9', 'J9', 'J8', 'J7', 'J6', 'J1', 'J4', 'J7', 'J6', 'J1', 'J5', 'J7', 'J5', 'J2', 'J9', 'J7']

최적 장비 선택 염색체 (machine_assignment)
  Job J1  Op 1  →  M3
  Job J1  Op 2  →  M5
  Job J1  Op 3  →  M2
  Job J1  Op 4  →  M7
  Job J10  Op 1  →  M2
  Job J10  Op 2  →  M5
  Job J10  Op 3  →  M6
  Job J2  Op 1  →  M1
  Job J2  Op 2  →  M4
  Job J2  Op 3  →  M6
  Job J3  Op 1  →  M1
  Job J3  Op 2  →  M5
  Job J3  Op 3  →  M3
  Job J3  Op 4  →  M6
  Job J4  Op 1  →  M3
  Job J4  Op 2  →  M5
  Job J4  Op 3  →  M6
  Job J5  Op 1  →  M1
  Job J5  Op 2  →  M4
  Job J5  Op 3  →  M3
  Job J5  Op 4  →  M8
  Job J6  Op 1  →  M2
  Job J6  Op 2  →  M4
  Job J6  Op 3  →  M6
  Job J7  Op 1  →  M1
  Job J7  Op 2  →  M4
  Job J7  Op 3  →  M2
  Job J7  Op 4  →  M8
  Job J8  Op 1  →  M1
  Job J8  Op 2  →  M4
  Job J8  Op 3  →  M6
  Job J9  Op 1  →  M1
  Job J9  Op 2  →  M5
 

---
## 6. 세대별 적합도 변화

In [7]:
print('세대  적합도')
print('-' * 20)
for i, f in enumerate(ga.history):
    print(f'  {i+1:4d}  {f:.4f}')

세대  적합도
--------------------
     1  0.6130
     2  0.5639
     3  0.5639
     4  0.5508
     5  0.6507
